In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawSales"
silverTablName = f"saleslake_{env}.silver_{env}.cleanedSales"

spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(sale_id)   AS INT)            AS sale_id,
    UPPER(TRIM(product))                    AS product,
    UPPER(TRIM(category))                   AS category,
    CAST(TRIM(quantity)  AS INT)            AS quantity,
    CAST(TRIM(price)     AS DOUBLE)         AS price,
    TO_DATE(TRIM(sale_date), 'yyyy-MM-dd')  AS sale_date,
    UPPER(TRIM(region))                     AS region,
    CURRENT_TIMESTAMP()                     AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd'))
    FROM {silverTablName}
)
ORDER BY CAST(TRIM(sale_id) AS INT)
""")